# 🤖 Chat - Knowledge Distillator

<a target="_blank" href="https://colab.research.google.com/github/WholeNow/KnowledgeDistillator/blob/main/Chat.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Testa i modelli Student distillati in tempo reale.
Scegli il **task** e il **tipo di prompt** nella sezione Configurazione, poi esegui le celle in ordine.

In [ ]:
!pip install transformers torch accelerate

## Configurazione

In [ ]:
# ============================================================
#                       CONFIGURAZIONE
# ============================================================

# Task da testare
# "summarization"      → il modello riassume dialoghi
# "question_answering" → il modello risponde a domande
TASK = "question_answering"

# Tipo di prompt
# 1 → prompt con negazioni
# 2 → prompt con domanda diretta
# 3 → prompt minimale
PROMPT_TYPE = 2

# ── Modelli disponibili ──────────────────────────────────────────────────────
MODELS = {
    # (task, prompt_type): "ID HuggingFace"
    ("summarization",       1): "Marchisceddu/smollm-sum-prompt1",
    ("summarization",       2): "Marchisceddu/smollm-sum-prompt2",
    ("summarization",       3): "Marchisceddu/smollm-sum-prompt3",
    ("question_answering",  1): "Marchisceddu/smollm-qa-prompt1",
    ("question_answering",  2): "Marchisceddu/smollm-qa-prompt2",
    ("question_answering",  3): "Marchisceddu/smollm-qa-prompt3",
}

# ── Fallback locale ───────────────────────────────────────────────────────────
# Se MODELS[(TASK, PROMPT_TYPE)] è vuoto, usa questo percorso locale.
LOCAL_FALLBACK = f"./results/student_distilled_{TASK}_final"

# ============================================================

## Caricamento modello

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Risoluzione del percorso modello
_model_key = (TASK, PROMPT_TYPE)
_model_source = MODELS.get(_model_key, "")

if _model_source:
    MODEL_PATH = _model_source
elif os.path.exists(LOCAL_FALLBACK):
    MODEL_PATH = LOCAL_FALLBACK
    print(f"[INFO] Nessun modello HuggingFace configurato per ({TASK}, prompt {PROMPT_TYPE}).")
    print(f"[INFO] Uso il fallback locale: {LOCAL_FALLBACK}")
else:
    raise ValueError(f"Nessun modello disponibile per ({TASK}, prompt {PROMPT_TYPE}).\n\n")

print(f"\nTask        : {TASK}")
print(f"Prompt type : {PROMPT_TYPE}")
print(f"Modello     : {MODEL_PATH}")

In [ ]:
print(f"\nCaricamento tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Inietta il template ChatML se il modello salvato non ce l'ha già
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
            "<|im_start|>{{ message['role'] }}\n"
            "{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
            "<|im_start|>assistant\n"
        "{% endif %}"
    )
    print("[INFO] Template ChatML iniettato (non presente nel tokenizer salvato).")

print("Caricamento modello...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()
model.generation_config.max_length = None

# Stop tokens: fermano la generazione sia su </s> che su <|im_end|>
stop_tokens = [tokenizer.eos_token_id]
if "<|im_end|>" in tokenizer.vocab:
    _im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    if _im_end_id != tokenizer.eos_token_id:
        stop_tokens.append(_im_end_id)

print(f"\nModello caricato su: {next(model.parameters()).device}")
print(f"Parametri: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## Chat interattiva - Inglese only

- **Summarization**: incolla il dialogo su più righe, poi premi Invio su una riga vuota per inviare.
- **Question Answering**: digita la domanda e premi Invio. Poi ti viene chiesto il contesto (opzionale: premi solo Invio per saltarlo).

Digita `exit` o `quit` per terminare.

In [ ]:
def build_chat_messages(user_input: str, context: str = "") -> list:
    """
    Costruisce la lista di messaggi (system + user) in base a TASK e PROMPT_TYPE,
    replicando esattamente i prompt usati durante il training.
    """
    if TASK == "summarization":
        dialogue = user_input
        if PROMPT_TYPE == 1:
            return [
                {
                    "role": "system",
                    "content": (
                        "You are an expert assistant strictly dedicated to abstractive summarization. "
                        "You must extract the core event, problem, or decision from the conversation. "
                        "RULES: "
                        "1) Do NOT copy, repeat, or quote the dialogue. "
                        "2) Do NOT use dialogue format (e.g., 'Name:'). "
                        "3) Write exactly one or two sentences in the third person."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Dialogue:\n{dialogue}\n\n"
                        "Task: Write a brief, third-person narrative summary describing "
                        "what the people are doing or talking about."
                    ),
                },
            ]
        elif PROMPT_TYPE == 2:
            return [
                {
                    "role": "system",
                    "content": (
                        "You are an expert assistant strictly dedicated to abstractive summarization. "
                        "Your task is to extract the core event, problem, or decision from the conversation. "
                        "You MUST adhere strictly to the following RULES: "
                        "1) Paraphrase the dialogue entirely in your own words. "
                        "2) Format the output strictly as standard continuous prose. "
                        "3) Write exactly one or two sentences in the third person. "
                        "You will be penalized if you fail to follow these formatting instructions."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        "###Instruction###\n"
                        "Write a brief, third-person narrative summary describing "
                        "what the people are doing or talking about.\n\n"
                        "###Dialogue###\n"
                        f"{dialogue}\n\n"
                        "###Summary###\n"
                    ),
                },
            ]
        else:  # PROMPT_TYPE == 3
            return [
                {"role": "system", "content": "You are a summary expert."},
                {
                    "role": "user",
                    "content": (
                        "Summarize the following dialogue in a sentence\n"
                        f"Dialogue: {dialogue}\n"
                        "Summary:\n"
                    ),
                },
            ]

    elif TASK == "question_answering":
        question = user_input
        if PROMPT_TYPE == 1:
            return [
                {
                    "role": "system",
                    "content": (
                        "You are an expert assistant strictly dedicated to question answering. "
                        "You must answer the user's question accurately. "
                        "If a context is provided, base your answer on it. "
                        "RULES: "
                        "1) Provide a clear and concise answer. "
                        "2) Do NOT add unnecessary conversational filler."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        (f"Context:\n{context}\n\n" if context else "")
                        + f"Question:\n{question}\n\n"
                        "Task: Answer the question."
                    ),
                },
            ]
        elif PROMPT_TYPE == 2:
            return [
                {
                    "role": "system",
                    "content": (
                        "###Instruction###\n"
                        "You are an expert assistant strictly dedicated to question answering. "
                        "Your task is to answer the user's question accurately. "
                        "If a context is provided, you MUST base your answer solely on it. "
                        "RULES:\n"
                        "1) Provide a clear and concise answer.\n"
                        "2) Maintain strict focus and provide only the essential information. "
                        "You will be penalized for generating unnecessary conversational filler."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        (f"###Context###\n{context}\n\n" if context else "")
                        + f"###Question###\n{question}\n\n"
                        "###Answer###\n"
                    ),
                },
            ]
        else:  # PROMPT_TYPE == 3
            return [
                {"role": "system", "content": "You are a question answering expert."},
                {
                    "role": "user",
                    "content": (
                        "Answer the following question based on the provided context (if any):\n"
                        + (f"Context: {context}\n" if context else "")
                        + f"Question: {question}\n"
                        "Answer:\n"
                    ),
                },
            ]

    raise ValueError(f"Task non supportato: {TASK}")

In [ ]:
def generate_response(user_input: str, context: str = "", max_new_tokens: int = 128) -> str:
    messages = build_chat_messages(user_input, context)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=stop_tokens,
        )

    gen_tokens = outputs[0][prompt_len:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


In [ ]:
def read_multiline(prompt: str) -> str:
    """Legge righe fino a una riga vuota. Usata per la summarization."""
    lines = []
    while True:
        line = input(prompt if not lines else "    ")
        if line == "":
            break
        lines.append(line)
    return "\n".join(lines).strip()


In [ ]:
# ── Avvio chat ────────────────────────────────────────────────────────────────
task_label = "Summarization" if TASK == "summarization" else "Question Answering"

print("\n" + "=" * 60)
print(f"  CHAT — {task_label} | Prompt type {PROMPT_TYPE}")
print("=" * 60)
if TASK == "summarization":
    print("  Incolla il dialogo, poi premi Invio su riga vuota per inviare.")
else:
    print("  Digita la domanda. Il contesto e' opzionale.")
print("  Scrivi 'exit' o 'quit' per terminare.")
print("=" * 60 + "\n")


while True:
    try:
        if TASK == "summarization":
            user_input = read_multiline("Tu (dialogo): ")
            if not user_input:
                continue
            if user_input.lower() in ("exit", "quit"):
                print("\nChat terminata.")
                break
            question, context = user_input, ""

        else:  # question_answering
            question = input("Tu: ").strip()
            if not question:
                continue
            if question.lower() in ("exit", "quit"):
                print("\nChat terminata.")
                break
            context = input("Context (opzionale): ").strip()

    except (EOFError, KeyboardInterrupt):
        print("\nChat terminata.")
        break

    response = generate_response(question, context)
    print(f"\nModello: {response}\n")
    print("-" * 60 + "\n")
